# English datasets: remove train/eval overlap and upload
Run on **Colab CPU**. Add a write-capable `HF_TOKEN` to Colab Secrets.
This notebook downloads the seven pinned source splits, removes overlapping training
rows using the existing coverage rule, and uploads the English datasets to Hugging Face.
All retained original columns and text are preserved. Evaluation rows remain unchanged.
This does not separately remove repeated training problems within/between training sets.

The base model remains `LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct`; this notebook performs data
preparation only. No GPU, translation model, API key for translation, or smoke test is needed.
Running all cells uploads the prepared datasets (private by default).

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/LG-AIME-English"
HF_USERNAME = "Seungjun" # @param {type:"string"}
HF_PRIVATE = True # @param {type:"boolean"}
DECONTAM_NGRAM_SIZE = 8
DECONTAM_COVERAGE_THRESHOLD = 0.7
DECONTAM_NEAR_MISS_MIN = 0.5

DATASETS = {'s1k_1.1': {'repo': 'simplescaling/s1K-1.1', 'revision': '96c411f1fe4c49d20f0e2a1565f61e1a28b0b84d', 'config': 'default', 'split': 'train', 'role': 'train', 'problem_column': 'question', 'answer_column': 'solution', 'id_column': None, 'expected_rows': 1000, 'translate': ['question', 'deepseek_thinking_trajectory', 'deepseek_attempt'], 'upload_name': 's1K-1.1-ko', 'license': 'mit'}, 'dapo_math_17k': {'repo': 'open-r1/DAPO-Math-17k-Processed', 'revision': '31dd309567e3da778038cc87d868b6097a3ccf68', 'config': 'en', 'split': 'train', 'role': 'train', 'problem_column': 'prompt', 'answer_column': 'solution', 'id_column': 'extra_info.index', 'expected_rows': 14116, 'translate': ['problem'], 'upload_name': 'DAPO-Math-17k-ko', 'license': None}, 'aime_2024': {'repo': 'HuggingFaceH4/aime_2024', 'revision': '2fe88a2f1091d5048c0f36abc874fb997b3dd99a', 'config': 'default', 'split': 'train', 'role': 'eval', 'problem_column': 'problem', 'answer_column': 'answer', 'id_column': 'id', 'expected_rows': 30, 'translate': ['problem'], 'upload_name': 'aime_2024-ko', 'license': None}, 'aime_2025': {'repo': 'math-ai/aime25', 'revision': '563bb8404243c5f09de6ec262f2db674fe5bce9b', 'config': 'default', 'split': 'test', 'role': 'eval', 'problem_column': 'problem', 'answer_column': 'answer', 'id_column': 'id', 'expected_rows': 30, 'translate': ['problem'], 'upload_name': 'aime_2025-ko', 'license': 'apache-2.0'}, 'aime_2026': {'repo': 'MathArena/aime_2026', 'revision': 'd2de22f3c656b4f56cf8981212186377d1e23bc3', 'config': 'default', 'split': 'train', 'role': 'eval', 'problem_column': 'problem', 'answer_column': 'answer', 'id_column': 'problem_idx', 'expected_rows': 30, 'translate': ['problem'], 'upload_name': 'aime_2026-ko', 'license': 'cc-by-nc-sa-4.0'}, 'amc23': {'repo': 'math-ai/amc23', 'revision': '80815d37005feb82cd7f8fbc6901d5d3eff43057', 'config': 'default', 'split': 'test', 'role': 'eval', 'problem_column': 'question', 'answer_column': 'answer', 'id_column': 'id', 'expected_rows': 40, 'translate': ['question'], 'upload_name': 'amc23-ko', 'license': None}, 'math_500': {'repo': 'HuggingFaceH4/MATH-500', 'revision': '6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be', 'config': 'default', 'split': 'test', 'role': 'eval', 'problem_column': 'problem', 'answer_column': 'answer', 'id_column': 'unique_id', 'expected_rows': 500, 'translate': ['problem'], 'upload_name': 'MATH-500-ko', 'license': None}}

Names use the exact source repository name (without its owner), prefixed by `dp_removed_`:

- `Seungjun/dp_removed_s1K-1.1`
- `Seungjun/dp_removed_DAPO-Math-17k-Processed`
- `Seungjun/dp_removed_aime_2024`
- `Seungjun/dp_removed_aime25`
- `Seungjun/dp_removed_aime_2026`
- `Seungjun/dp_removed_amc23`
- `Seungjun/dp_removed_MATH-500`

        For example, the source `math-ai/aime25` becomes `dp_removed_aime25`.
        Only the configured source config/split is copied, including DAPO's `en` config.
        Existing translation repositories are separate. Rerunning updates these same destinations.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "datasets==5.0.1", "huggingface-hub==0.36.2", "PyYAML==6.0.3"])
from google.colab import drive, userdata, files
drive.mount("/content/drive")
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Add HF_TOKEN to Colab Secrets and enable notebook access")

In [ ]:
# @title Install the bundled, tested project code (self-contained notebook)
import base64, io, sys, zipfile
from pathlib import Path
CODE_ROOT = Path("/content/lg-aime-english")
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = 'UEsDBBQAAAAIAAAAIVwAAAAAAgAAAAAAAAASAAAAY29tbW9uL19faW5pdF9fLnB5AwBQSwMEFAAAAAgAAAAhXHQtxyXcAgAAhggAAAwAAABjb21tb24vaW8ucHnVVUtv2zAMvudXEDrJa2psBbZDgOyybodh6AbscemKQI7pRY0jGZLcJCj630dKTuLEwbDDephOJmk+vk8kJYS4bp0qaoSPXz/fQKVr9KBMCapp0JSX1tRbuLetM6r2sNZhAQ7n9gGTkzYBnWubgCUEpWufCyFGI71qrAuwUH5R62In3ntrdt/WjypnV9CowL9Ap/5C4mg0KrGCUv9CH+SDqlvM4PIt+OAmI6DjMFA9u+i5X6ir129kNPHhNHnZrhqfnMfgKfRsiVs//eZYRuNbhzPl51pPPxAwzHI0c1uizGKYLF/gpisg6+pZOx1wxsElFz2GVFkqiTUwjeVHa7bX5o1yaEK+WpbaySTs69hoH2Z2GcXkEpCJUG5L0aI7Uz7zbVXpTYycp2+4AJGHVSOSW7yYvW9u6e6kWIsMlIdqMqRmx0x1jowxXWtJZU6vsr1nlUf8Uvw0oq+t6tYv5EFjfV75rZlLMlEvGSuzE2C5w6ZWc+x4OiW37th1du3/L3Ir67hqIi8WfzD0+es1J/10thU5+/OwnGb6lOZ/zXKkK/olptSQqecnIwF2qI7hviDE2CjtZrytuhx/wq8rMDYkOBGrl9kBSLeJbu/SXqJbpwidtKJ1QpJwF4XgML28gJSWLIU4Rxg7njK2XhAyYI6P28oHRVtzSqwErOseKXxqbTDamAYWTuwdNrYcR+VTkNPySBvcdvgbN/w07ZXaqtJLjnacBjdzbAKk2+ZX5hp51b53zroxfDeapZ4uG2Y5oY/fJ648IZOvsr2Gs9MqLz1zKovYOsNwfIgx15q5ojaMJGZn/xqyEEErTff3g1dorFhW4p2Nz2B8RT+BClBsA8JjDP00gUe+3icahPjo3ViDR0G5c/I0nTwJg2s6B/7voFa5R1zKl2O4GiLczWFxGLGuo7mgboZqoohGvNjOdJkG4+gVJoiOa74VuhR32SR2RH8Vno5gf/7i0nga/QZQSwMEFAAAAAgAAAAhXAAAAAACAAAAAAAAABQAAABwaXBlbGluZS9fX2luaXRfXy5weQMAUEsDBBQAAAAIAAAAIVxPMocyhggAAGcZAAAUAAAAcGlwZWxpbmUvZGF0YXNldHMucHmdWW2P47YR/u5fwTIBIgNeXfKphS8ucEC2aD4kPVxyRQHHELgmZTMrkQpJnddx/d87wxe9WNrbvRo4r0UOZ4bPvOsopfdPTSX30q3I/ij2j4ITq1uzF4Rx1jhh7FuijTxIxSpi9MkSZgQxwjGpgPYk3VG3jtStY05qlVNKFwtZN9o48rvValEaXZO9bs4krnIhGnwOOw1zx0o+pM338NidP7O6WiQGdQ3cpe64yIOwoLQRjBcoqFqRk5FO+Ifhb2Dxw7v3/yref7j/x4//IRuSLQh86C+6+iSIOwpS6qrSJ6kOpAbxpDH6oRI1sU405OHs/+bkVyCsmHWkgosTXZIzwATibaOVFYQGphbQqDh58BSBt6nJO2VPwqzJ1+EHyRJsf7TaCbskp6MAVNO2tP5oZMnCotN+MSqX/6Z+U3SxXCwWXJSk0oCCEQdpnTlniOmG7rUq5cG+4cwxK5zNEU66XHuuYMDWKI9wblkpCuSQIfr+9DL3uDrx5LJlEqIAcMEzcIIVeRTnyAkuCEYEi0iFq7kFd3IZzZMkL02fAHf43iLlbqgBLEb2e21EAT6ni3jH7BOrWhHZyBJgkco6pvYi7KzAFNYNxABNJVQ8Rv6yId+B6xL/uP12lx8E6GV0JajfpK0VhvanvVJMgi3/jSfujdEmox+VEXt9UPJP8Hb0I1ILa9lBgIexphHmLVzcNmLvUtw8CIDEG2ovBAevostOhtcFoEg6bdFKTihHd+mWSruZm4Jdh3hO1PSagci6wVgELwUPROshABr8Fe+aNI/6gCzPPAdJxll0yWwQKANxY60R4iEdWQfdRTXH8cUwo5+910f1qPRJBeQRFNPuMc/06BtRthaZQoAcWrhhvF70L69Q9DCf0QpMYpliNaCKPwFbsF5UIuQ8DnfdhktZIRQ8Qfxky87d5apPieD1QrW1MMwJDA07uE5HtOmSXpbWep+QHOwvSwkGSrkpfWLEpTNB1S2VvNjrqq0V3S1H9GCACcWIQFSAb0kveP3r+uKJv/ER+80OnuWVduTLYVgNdITk9DN6FOAA1sj6nSVigXi9EFIl/UlabzFgwVusPoBdih7JkU1UkFx69tdBGKGUnHGe3WjQU6QcvumMsA3IxI0Ont3wmiiVbCA1+DSEXlp899fHmxzRs55mrPh3hN1NQEeSENIIAe6nrA5rssmWr0Yw6dJB9uYZxGIRmcAR1ufRYF01SgZHVRHysLP8YnUTxxe1DWGYY4grPo6JCzg3XQ88ckWSUWG5QzfeDJbCD1hJV4e19PN64+4xaUT5MW0EzyxAVsVtKH+DlBGPXDpOAcu1L3edZjsENOx4B0vrISI9aafebhsIe2Ng0omHMca85ZyBi1UQOTHEr6mMMujjiqjznhkecx2eWoXdFeTKR6FSAcf+6tgeQPihZHDo2Hbt2LHEp4JDCsYGYeEPcMhKoKhPw5vAkbzpsspQdF5zmmrN4FQunqBw26HjBBQD+yjMp+EbBXpXCCAY0Wi6A9t+uH/3w0/3KA9bQohId27Ehsbexy9+giDUapMOhkc87MHY+O9F7wq+em58Q5r1Kg0bI6H2Guv7hrauvPtb9N+vyAdxB3dlD9CdHH3hGwIm6xqaAjBbdU6dAraqwGZFWAn9NiR9d9LmkeBXHlli8ykVkEExqR+5NIRVGJMQm5BY2rKEDh7iAXFGgbVuFZYxqLgcOHJpMZKYsfmtAXNoyOBg7plm4cFufjXYc3grFfrRPyZQ0C+YOfuS1nPBUl94PZ6g9at57uqG3hzJQ0vuocMvEDALYH8ADFmBQ2YDSakXTRaJ3gZhGZtfLOraJaNi4hq6eeqFk397FnHxxWD4Z/mukSEEgtNH70CBS4gAGnWhPclrcWWNBG5eQDZwx7CJ+q2gZ1KyBCCA7nJdwb+uHenjG7NDAiIHsOtRjFl5ABBbg4qH8SnzeawjqGGqQ2EFjgBzoZ0Icpyr+lYB257nznjSqqcFD+3IuzwAOZqPpc+kiKQhCEGeOdrOZuNTo6lleBDEIuWWdiBASoYhoHscy/G3mtaxX0KPEiYrmJeZOvgoM7GevcU+mzAI4RNWot9xKkDvoGNl/Bi9GYyuWQfJROtoKN9ZosLhFnFqKMIu3X2R9t5Cfti3bU1KJqvRJQbKYnEas5aq1KA6+Gsew6bApWycjp9NtuPrcURhGILZ5BojvpNd1DfKCDaZI/LtbaTyv+eIni8PE9JBfI43x5drjFQu63ptcuE2L4X3NQuOggbdXHCW4nZ5pRO7xx3vo14l8YRDpuB+fnmlxT+qdCq11xbm2IGtQa1nVPgKOUmOnTlWFLZ3LdQeC25Ts1S4oKhUZz96Yc2BJ/id3wwwItTxy3z3nSaamzb0Osfkv2EMm3Q/S3IHTWFqqK63QE4GoilyQUAYKG6pB+8zYF6fmAk74qjf90E/MHM4WyDCdvkKQ00o8APxGsCupYVJZH8cma2OHfUlCr+blT0Y59JnNhNNBmOexuIRNbZ0QI0Bn+NvH7a505CG9pBx/aA42gvd7WViE98b+tcxMPwJZeMbmQB+WvqipBYPzebk9H7mcy9mwvVe3TnPFaXLRN1BsVn3lWaaUUJ+W7+Q7PqktA4w2yOboYqJMHF7Pi/GZJgIn02NNGUtIB3ksN7uc6pijlr7JOYr1wxJH2PraYzO0B+E8i9ZeAF0pq0QU2zvJpQvvgTBz+dfhHz/pzD67gFKEr8D/e+k4uLp73RGq+Ss6zmfnrfOsHCvR/V9TD+OmsH77L5bCG+wls8QjnujlffT0ZuJzjyvrDIz4fcjtPB1U4n+BU5oL2bbCRyNGobDS5e/fEoRn4TCV+Aq/gfDoBkkJ3wxzirslc6BN+9LzP8Tsch1i2S7vqcOz+EVtQ1ALW6wTH0tTWdCD9zzGL1AGLfsi/8BUEsDBBQAAAAIAAAAIVwXRcfZ1QsAAFMoAAAbAAAAcGlwZWxpbmUvZGVjb250YW1pbmF0aW9uLnB51Vptb9s4Ev7uX8FyD4jUKqpdXLcLbb2L3NboBtcmRdrrAev4BNmiE25kySvKSdPE/31n+CKRkuyk6KcTkNiShsPhzDMvHJpS+iHh5Q0XjLDrJNskFS9y8tPhRZmsyKK4ZmVywX4mRZ7dkqpMeM7zC1IWN4IskpzMGSnZCqjSkFI6GPDVuigr8qcocvO9ZOZbxb5UN2WyHizLAnlnGVvgbIJogt+KTV6xMiApWyabrEr5olLEHB5XRZHVpF/5Os6K/IIJTbFOqsuMz837D3A7MBOtVkUe8sK8SzkOC0CyJI1R1CwgNyVMIW/s79lgcPL27Oh9/PH4jwkZk58GJ6dn74/eHf9x9On49CT+PDn7CJ/w5sXgt1O4O3o7iT/9fjb5+PvpuzfweBi+GpxMjs7i98cfP8K/E/ns5WAwgCWSvChXSca/Mg9V40cDAhd+BSr8CLPihpWeH5ZsnSUL5tF/0IBQ6tuEJQvFZu6V9Pzc+zXK2LK6L/nFZXX/1yZJfe/XJ9Pk8OvMvz8/n3r++fR8Fvz8ZCbZBJJFPzNv+r/zm3Mx84GqpOR8RBzyklWbMieU0PDPgudeVVyxnCyLkqhvPFcLEOuMV55P+JIk+a23CLlIsnyzgkdIvJCEOML3tU4Qd0LxEwHJ8TYWoKJxYwitKC3DXbVZZ0yPmHISEU6eWQNnaiqOU5UJIMbLmBZY+OTQooRhI3+r5QBf4GkCOBCsqgDzwmsIg9ox4uqyZOKyyFKQlSVlvOJCwL9cywjr5oLnokpyMJ/NYQ5o9gkIlhfVLhqeV4qkEfE1GSnOUgMJuu1n8Fo2Kcui9KhFudqICv0zIetC8IpfM+THLlip8cOXcu4heT12ZYdJuutDqr1Tn7G/Nrxk38SPGqunbFHkVQLEoHJPzgLKTwJt6AsuqvJW3T1VH73IUK+6k427zqm52HKOHU8NBtqG34sEpew8ZV/AwazI5gE/9RKGlTIeLVnJAAQAfDnDAuOhgFHTWUDutvgn6TFQozTyVZ6smES4/ALaRsWhcY3aJMVsSssiY3RGxmNCkQGdSV72yJpvY2d8DdHe8FW8mtd4XbFbkMPDNwGBCT2gn1Ke0pnvO4TN+qYwBiQhd857vKiRgUZSrGAHBbCPSD3RLqp1WcwztjKk5rZFv3Xu6qCcgoBNhHYZuAvTntSMjDrydPxlSSerdXVrT2flXz1RRO5AU1vqTqdily1dagJtWyw71L22fKYroMRgmKzXLE89r+EcoH1bhmSZ6OEgNS7DN4i2K4z7nWEW0g0sUOqGW3cIglLWJwazkq4rkVQCOt4UCWZhkqYerkZFiYwlOTod1gTCclqG8ksHRcT7yvOmrrcEqgTSXhFCvQCr9RsJdrvfE3A/WUdRV16MfzzfsPrhFVvLmCCrKy2eVJOOB/g3bOIx0oEm6pChaykLEpYro/DR96L+BzKB0GdVhcokAnQnYCmLKiSTZHFJFkoSfFNdQgAAORgAnC3AI7LbFst7w00jCVNWKaBOtAx9L5dS5AxL0FTGZvnWeE3oeoHhICvNPsW0IaWw23UuB8ddtDkThZs1yuVJ+IUXrPJwaEA8v+VL9RJkMO8w7UZIvJ4+bcXSbgDEi65YdVlgpNSFgUlWdAd9/T5SViPPO+65a6akWlyyNM4besNkxwhpT5f84bm2vbEAaAMtMJhPgD+z1HPNoT3U1f2sAwEofsFnAX4y7EluGBd7jL1saJFsX+zHqzGzibG9ZHj1m9xcjzS9uSwIyJXE7EuyqOJa9h1IqIdbiBiFw4fm6gXBQ6N6gPDQEBQqS9ZA2Rhs54guZvDa5YUhwgezxDhLVvM0IZCEvcPVtNEExF24reuUmSp6zCNV97jclV6kg68kzGSYsTwf0OTMQH4Z9xSWLlxxjOLbU23oPfk+qO2GGX2ouKoJH1dfOUPmsPWOLVTpNUyHM0fFuxnoAc3Qftqu1bt1hJs0Tci+c6zb2EtPt31MLYTJ22gfVNOdGqxn2R++tbdMY+t9Szm9+6n+sGNVDc9gv9VLgxYhe6bbzVhVS98Z0Gid8fdV/c6Ix+JTEn87RuWwNk7x/lHhD5H5WDeHvRjy3SPLQ8FLVrGqxAQbIuysklBWtvVL1wy0KPkFbLezOuhi1Y2Foe9KQ3UwcenUsx5SbUubGKXawRStoitc933jDYZTA+U+VrUXx/Pb2NpD3uVRy82nufLp3NnxbhumW7u7tGeXoFsXUMmuk5LFdgsDSo79TYyyKKr/53aGGevAipoiSCogBkYCPsEIvU3TwBpXC4B2bqRpKLpiyeqyI6vF0xbawMfcKzplaMEvwGIb2JOMdWfYu6PGWhKf2nCEojXhiTQqFGJGCToj/EA+j3CTkzMQi6Bs2EiuWB6SdwlugBaX2H6EaJ+JAmEjWHnNcE9ElryEGPz5BSk3eag3mlkKQ8ayke0hXHwoxmkLZPH1CyrJ54lgsQIpjNGDe+gVSYi9bWqagNbYkH2BtUKdLDMSUoVZkaTCs2lk2xy7u1BOQwTO15sqVopT29taoXYXSa+mlqwm0s6BnGPs4X+b+DJ3xBjh+nthxngP9MPUhlxlOhRC4hrJokaYJZVPto6jQ40lzwmoM6sl1dZo2Vqiq+Uky7x18wj5rJGJlCPElhDDjYvVejVmtuxjc7ft4zYkkOaRFpMTdXq8b1x7QG6DhGd5UFqAHbAVJlMh4RU2A2CnXxXgSq2k5TRZnCaFarBIMNR66HRYaj51fmuOczwc0+mFaee2Bvm4eKOWYlM1ehF01tfq7NXKcoda4G9xJTZQCiQ8g32hQpTdzOvLMY/pTXWb5Xg1uUbxtvMNXlaK6YuxPWlmX4B1U01PdPUtz3bSRBM4rXDtgDJqIGXnANRq3OApkvsu5Z8A+izBM464KlS0tAoOGbcxyctorlTakyhk5WbdWTQtbEQmOPRBalcgsHPOo1FuHUVKUAc25pWGGxI7ChjwOMd1LtBM+cJzHMJLry7EdY0cKDfW4kjCBm1LSsgEohPoQlJND0wtdTDbPnee8RQekfsaYWPz1jw4mEXhP5dbQi3u90R1LmpidQucqAUvJRTKcnfw6ezo+OT45C35cHb6r3eT9wfR61dDnHfy+ejdf2TtYV5pH8TDUn3UiufSIf7r0cINT6vL8auh1iUernZGKSHdU4dZeyiaHSqHORrcOsD2UI5AMQ6gFsgyGfXHlFpIsFaamIXdzbf1CZqyY3/C1MDYZchW+AKL6ph40MQJ0Lv8SYBoSkRsUtgWq0d1wwaMDohTHZLrZpK+qnGHnU+lIwpLxFZUQPBRv+Vk2v9VQaCifR0SZl2naylIre48V+E7staM1zMZhR7Zt72KjCzTq/59tWxnoqDdVrC52ju3Pb0Sd+u2n9DZuO0hbe/M+km7q3N3sk0S9Ft6p2dK6NavTaSD4R3DMwXZ7keTevIYFH+KonuCfkTdasKEJatDbGywb+vYg4tGRumGhjR6PXq5JXeSR/TLS5PgxYiM++CGZwFUjK7iUTiieLjUoNVwlPanCV+x+MXwxUv8sYa5+ZFaJQlG7e6xAW7iTeh2QdwcAImRksNsw/E4qUusq7hcdU5Me6O/Den0zKCmNnc1cTPMnGJhb+DOPTA2MgYkVlkRFrhtIwSdUYz+fQgKJCLJeXWraq2oscnWtBW0XbYKRB77soY4AY+Hfkid6lhO1Wdr+t+jM0wrUG7oOWHrpvo51SVs9zB2QhAPyXEukDlu6KBA1HDEYo1jQBMR7bVG0Gi5RwQXcp+MRxy/wQgIWjuQuZV2+4tWWm+1vWxkzIsSiqeM57j51ecmNaupV590+nYwFfvjqPOTATGtITazulxWex1HRcTDj+kIdmVu3w3ElW+Gustek6k2u640p9FoOLPyBOBjNKwRIC1/w6HOwq02/qRK/jBGVIeonjqdRZ20oRfRKKmbjU1WeG5Zg3iyo9pUO/KVsy5d8Pi0Be1+g7neB7qwc6KVgM/zE4jNRO8TsOumvTjfQLQs2TVnN4Ab1b/Tubcpdw9mvl2BUYiszNl4oD1378fBn3S+/htQSwMEFAAAAAgAAAAhXI3JdN23CwAAmSIAABsAAABwaXBlbGluZS9lbmdsaXNoX3B1Ymxpc2gucHmVWktz20YSvutXzE4OBmMKtL2b1BZdPLgcuZKDk5Tt3YuWBUHEkMQKr8UAkhWF/32/7nlgAFJWpErRePR09/R0f/1ApJSfu3SnRFploumvi1zvRaY2ddWlZV6lncrERbXjx219p8Vd3u1Ft1d5K+o234GkEHqzV2UaSynPzvKyqdtO7FO9L/Jrd/tfXVdn27YuRZN29ELYF7/j1i+6T8vizJBt6rKsqzivHWGW75Tu5uKuzTuVML+zs0xtRd8UdZolVVqqSDdqM1ueCfy1quvbSsisSVpV1rcqS6R4KYjiUraqqeU6bnVT5F0kF3IuXs8uz1+vLc/btMgz7D3ZFCqtIlymc8HXczDe5bpr7+mKNLPy8q3QqmPSmfjbim8c6Qym4gfM4vi14cBap7lW4t9p0auLtq3bSF58hcZ0CupWtfewS7XNd32LByQKbOSMV28hgmww5y2KvPKKxjBYqaNACulq7FAXSq7FaiWkwp7lQGLJWOFL4rsmrUmkuRtTnlR9Ky/As0+7vK6M72z2abVT2VI8EI+D1Zz+VKHVmKU9NLESD9hE1F7KPJPrGe+zNdsj619KawYt10YzOl1eKteHEUflLLkSl1DHcMK/4DXsi21D8uo7J7GqOyOPua6fMJGXAu7GZ6OAgs/eaV73XdN3iaHy+v8ly74nlsaoWa7TXauUicwwdNnuLOuExaE5bSzQbWL/Y6G/1kYiLJHCIPjPcH2LS/KnjnDBShTXCvZVNjoh1wSWPaxkk7ZZNLiriyUEWd1XLqS2eaGIBge25WNePLDbvuCwfbE+nL+iv/N6yxev4yZt/9cjInh1qbqUFpEH+Z3JAh7YA+/kUlxKVcn1fHin677dqMQ7FEhCuAhJAZn8XpbAL5ijyzd6xMvEKdOMzPpwdLqWlAEM5EaieRZy9OQwY9oXHUi/tL06RQD9E7IdS3+QbC3P2dyt50ISEuOxs/JhImwIHvvi4HEOjOIdAEwW+UZVWskAW5zZL/1L4IuV7Z8wdae+dngVDUY7Pz//T0UoTYkg1ukWh9GXTeR4wlngJMmNuterDykgYwZaswrrPJ+t/E48HKWFw5ToMx83mch4FZ0ynGod7buu0cvFYt/vkN9223Sj4k29cH6xGNPPpnx/z6sK8W+8CY59m2tE4lJc+XXmCdZexeI9n/Tw1py8eTfSls5tIHMhAKpPsE5OEik4EekcQod4rJZ8VxSUEwdKzvg+hW/qoi8rPKSYrSxQzxHYm6LPYARfBDRtfV0gn8wD7RA5RU9woymQU+RmWtG1MByeYEXZdEjcadOo1si1xvnlJx0LwApIK10wYo34sovAzmnezkVZZ6oQO1Wp1lICYtMss6nQb+FOYQ+QVeQqC21IhQ5Uhz5gaFiQArzlFrDZEidt3qninvDtCrrfIjoqeACVHFeBVWdhOIxSKXaTV0Eu1X1ZpsjdAKKbpbjlxHMzx8U3sphL2iTghpKGz2pDXLJ1XoYRZFzlC8nHCSzUkH+xtC3SxmQxnHevlUYCaEtUOn8YktsHq8yL0fMEK627hmdjRCH5dPCRTgxrd21aJjr/QxFA0w0OBiyoxsTxi+6EcuZ8p7zJT1KklFbpfV3AV7eDFMcy8a8hjYLJSuLtUTYKpIz5S+vIIjWEmapqzpp1G4vPe0phoYbW64kxUnyKHTsjwWkmnHV/jRKCQgCJYbPHBQXpFm4JP0KEvvvl44V48+rNP+b0+wP//vhWvPv4nq7+/lZ8fPfl5/MfXr2Kp5w/qUZxSe6sOChG2R/+RCGxaWutBxLnWuzolPMDH7dONQELizjGbQlzyPdjgmIdWWeeHa4mi0w8jAu50x4qv+xzHVqXwUzkBA0D+MBtmns6dDodgxex+ILriqob2rna5l+nFspxjl2+ze3xAxMKFCYQ8hZbD2WaZkZx7J+ywezs9AZs2hA2ly0DBV6K6ERqpCORbHa8I80ycX0fbIoVq9Sd48h20Fq1OOc4KNiQ6UhDssCA2iijiN4DO2HW59/+9en9RfLp4t1PHy/iMrsagyAqxatJlZiYsDIQZ+piAxNAb+fCp6JhhNU+yjWgVBmY93pVKm1FmWNbOh4DqO3TyNC2RqSCJKH+MaIKxZYWe6Cn7SljvU/f/PBjZNZz1UtdpKGO60ZVkWyvYXhENuJQpeXgkLS3zb6vbshSgNg2KtLyOkuXljLGTxa9RmyK7wX9M5uLaxnWN6xM3DfUHkbMarSPfbxXX23d74peTQ12okwCfayZtMVnmVb5lhtdVxG3de0LYuqMfTjbxvgncz8XHwANaAz1mdEHy2Aztgyz4KfPamx5xXfUA/AakaKEyCu0LNpV97Qz8g1Y1dRbdJNW907J+NmtKZV6VC+OrWFyYti7EB0V5qaUktx6uWLdP4NcQ0feDlywrZahtoc0tH6zJxsgG/pOK3JoDo/H2ys2vVi43kJLXG+loU7sJimIEaZyFuea6/bopCYf8ObXuvuA6i5zCn1ERJHJbTnFcEDWdn1Zq9q+cm+z+q6yzRhx1fmuYocR3hZe6oM0LoD2wLmh38Hy2FUlXBF4AAszPoCE3S7h3STJjKMqoRCPZocg9rfI2ygJVsJZycZIgiYaQtlaXs3L5ZtXxgUgpyWARz11eLaDZYoqFoNjK6fB4niK5BfYppKsFCxdmD7Lnucz2tKAJb3AXuLyJsvbyNzoFXd1qDGgf1Lf8O2gjEN+bYcYl9I9gVMHA42grR/CZmsBAmsdVsQEKQiLTRedDLlL6RbJ9aBFRiws7hgOOLUu8srNvaiVuxhFRqbjrjZreCDiFz4Zgb+5zBfOk0TWc7VlxpA0IrMl64m4NKLtAUQ06bHXs4Emmhy0z6QIUTN+ZFce6frkYANNdpTpGfIJqsCagHIl+257/s9pyXFKg2lK94pc33dKjzV5HuRwZBousxOKDNPWqUrfKCHkOINM+IzNNmY6abfkeCJxYnZi9nOMScZ7TwxHuE1bjnq2E1TBlIiGRCconMsmtutMUI6iZqMG7dHBjEG1xCMa6eGuT6lqi6ckaFiTPOOpjh2HDtPQMOC/xcuHmsuFSwf+/s1sMgean/ALC8F27LmaHI0M0BQCjrB1LMANu749+3pkjDWmefJ0JeEGKGw0TkUYJwpphupgysh+VDhNLM4fEWFLZ+s2XtfJ46nWaaXRqkwXjZ9O17gJ4HHUMOoh8mngcgswq8PYns2WYQU+O1pNLteQy9EsTmXh2rjdFfV1JL+Xs+N1wP1mqHAeczGT0H2FNZrdPhE40iRzybV8ZG4C8z8e1DKY+FrHDieeAXb5ckE6DR3guftRM+Ae2j7Afl7zncBQQPWADpM0mja/Ra09F119o1CUp02++rWuVNgBBHPJZN/7T2k/b981+aNUcd/lhW8afB9AQA1QORvXY1y+eSx1hl1PvzSRsXhONpAOc6zj0msqFPn8we38sHhgbpcvArRALRWkbtOXGqdF07JXmxvdlyTerLQef0Kwdb/Brf052pUhXq3x2Inh6sRJ+ksfZvhjasaipp+7Fg+Ora/MA2S3G8V5w/70i93yiUbsCSv+9TTx3b5G7rVxZB3LfC2blsTPPCQ6G/7g8vTR+DWk0KZV7mgj+jE1QNLdN2rlBErv3yvv548UusTSNNnGW7AaKLPTz+A9MPuOi1X6+ssV4oL7pJfDl6pNXSpBXSONh7t6p7q94qyKwOMv0XnHs8tRV0t/9t3KqsvWMb41LnOsw69Y+eM3k42M53DMLqHhxmrAtUdcd5Jt0LDXd7S0w0nqFRfc41iZLDD7SUqlNdx4JX+3/0OAHzi5rwBmYmf+XwCaM/I0dzpklqcqB++rQ+1wNBRnHY8n4Ebnw2Os3FDmyL4r5zNmMLAyu4zrPAPytsVqK5/84kMcDqcq9ZPJwZyJdrnB6xlYAQ1LR/jHjIEQNsKomkBomfbmT/qW41R1SHiqkh4kuwbatSO6B/W0lp4Ua98sdmVYfIYJeECT4em0BAnyqrdBkOrnx2NAT3b2f1BLAwQUAAAACAAAACFcrlDSu6ECAABgCAAAFQAAAGNvbmZpZ3MvZGF0YXNldHMueWFtbM2VyW7bMBCG73kKP0CVcBflW4CiCFAEzaG3ohC4DB3WEqWKcpa379hN7MhLkKBAUOggcEac0Xz8h5PpsqbndH42mw3Qd/NZjm3fQHamiWlxkenXAt0b713MsUvzWaWcoDTQAMKJyjMSCDBDpZJBUaCGaUusFh43uS6FuJjPPASzaka05L6J43w2DiamddSugd2qHzrbQFu7rlm1mOn3CvKIOdFlUr6HYevJ+H7yRL+1plXToAUeenAj+Hro7vN8RgkhaMUkKTdmxHw/ngN/wj+DPgMs6/E2piWWXON3v3B7Nzy+8JpxhLYff2KcVd90xtfJtBjpiU+x7NDTRAcpo7WN45k3fVe3Zrytabnc4e16SMVALz5f3nwrrtFdoLu4GToHOYOfgObUe04qqUrg3pSlJlw7p0uvlbaKVKXhzgWlX4CG9F7GuMbC3koYHjBOHVPozmPy8HCENmpD7eF+ynlAbwphynBzlia2UDPCxA7g1WqxwGP6YhxciYvtBxNwLIDWhgVKKuolEdqRwJWxyE4EW1Wl5d5XlXm7QuHONMfhrdeH9P4up+yiP6TF95V5CtW20NcwyR2mtfIKEzeAmJzQkYpb7E8imOBOBlJ5UOCYYoF5q5AQSOugsq/Swf75/+DIPTimN+4WCnZOtojUDtFaeJcDJLNVkZpw8swDY4E7JZUVQSoXdKUpw0crXpaeAuPW8Y9V0fPm6I8037uJqT1izhX2sUiuyKYQa2ytY/yIqtbmCSxNNJWel4TIAFYz58ugg3WqItiCnkMIghNZ/quoTo+Et6pKnBoGh5DWVR7rt821Lgk5dStdX36/KuRm6uwIKRDgcTqasjJhfYdzYpQtlQanZeWsNEF4buFD226VIpZfH+MkD6bmKTE9l3sM1R9QSwECFAMUAAAACAAAACFcAAAAAAIAAAAAAAAAEgAAAAAAAAAAAAAAgAEAAAAAY29tbW9uL19faW5pdF9fLnB5UEsBAhQDFAAAAAgAAAAhXHQtxyXcAgAAhggAAAwAAAAAAAAAAAAAAIABMgAAAGNvbW1vbi9pby5weVBLAQIUAxQAAAAIAAAAIVwAAAAAAgAAAAAAAAAUAAAAAAAAAAAAAACAATgDAABwaXBlbGluZS9fX2luaXRfXy5weVBLAQIUAxQAAAAIAAAAIVxPMocyhggAAGcZAAAUAAAAAAAAAAAAAACAAWwDAABwaXBlbGluZS9kYXRhc2V0cy5weVBLAQIUAxQAAAAIAAAAIVwXRcfZ1QsAAFMoAAAbAAAAAAAAAAAAAACAASQMAABwaXBlbGluZS9kZWNvbnRhbWluYXRpb24ucHlQSwECFAMUAAAACAAAACFcjcl03bcLAACZIgAAGwAAAAAAAAAAAAAAgAEyGAAAcGlwZWxpbmUvZW5nbGlzaF9wdWJsaXNoLnB5UEsBAhQDFAAAAAgAAAAhXK5Q0ruhAgAAYAgAABUAAAAAAAAAAAAAAIABIiQAAGNvbmZpZ3MvZGF0YXNldHMueWFtbFBLBQYAAAAABwAHANMBAAD2JgAAAAA='
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as bundle:
    bundle.extractall(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print("Project code ready:", CODE_ROOT)

In [ ]:
from pipeline.datasets import download_sources
from pipeline.decontamination import prepare_decontamination, print_decontamination_report
from pipeline.english_publish import stage_english, publish_english, upload_name
ROOT = Path(PROJECT_ROOT)
ROOT.mkdir(parents=True, exist_ok=True)
data, source_manifest = download_sources(DATASETS, ROOT, HF_TOKEN)

## Remove training/evaluation overlap
Remove a training row if it covers at least 70% of any one eval problem's distinct
normalized 8-grams. Eval problems shorter than eight tokens use exact normalized
substring matching. This is the existing v2 rule, not the old one-shared-8-gram rule.
Normalization is only for comparison: uploaded text is untouched.
The report includes per-eval removal counts, near misses and borderline pairs.

In [ ]:
clean, decontamination_report = prepare_decontamination(
    data, DATASETS, ROOT, ngram_size=DECONTAM_NGRAM_SIZE,
    coverage_threshold=DECONTAM_COVERAGE_THRESHOLD, near_miss_min=DECONTAM_NEAR_MISS_MIN)
print_decontamination_report(decontamination_report)
print("\nUpload summary:")
for name, spec in DATASETS.items():
    print(f"{HF_USERNAME}/{upload_name(spec)}: {len(data[name])} -> {len(clean[name])} "
          f"({len(data[name]) - len(clean[name])} removed); {spec['role']}")

## Prepare original English rows
Stages Parquet files with the source schema, original dataset cards, provenance IDs and
the decontamination report. No `ko_` fields or synthetic data columns are added.
Source licenses are retained; unspecified licenses remain unspecified.

In [ ]:
export_manifest = stage_english(data, clean, DATASETS, source_manifest,
                                decontamination_report, ROOT)
print("Prepared files:", export_manifest["folder"])

## Upload to Hugging Face
This cell uploads all seven datasets under the names printed above. It verifies every
staged file first and records exact Hub commit IDs in `uploads.json` and
`english_dataset_suite.json`. Rerunning safely republishes the same prepared content.

In [ ]:
uploads = publish_english(export_manifest, HF_USERNAME, HF_PRIVATE, HF_TOKEN)
for name, item in uploads.items():
    print(item["url"])

In [ ]:
# Download a small record of this preparation/upload (data remains in Drive and on HF).
from zipfile import ZipFile, ZIP_DEFLATED
export_root = Path(export_manifest["folder"])
archive = ROOT / "english_preparation_reports.zip"
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for filename in ["manifest.json", "uploads.json", "english_dataset_suite.json"]:
        z.write(export_root / filename, filename)
    report_path = ROOT / decontamination_report["cache_directory"] / "decontamination_report.json"
    z.write(report_path, "decontamination_report.json")
files.download(str(archive))

Load any published dataset using its recorded revision, config and split:
```python
from datasets import load_dataset
item = uploads["s1k_1.1"]
ds = load_dataset(item["repo"], name=item["config"], split=item["split"],
                  revision=item["revision"], token=HF_TOKEN)
```
These are English datasets. The previous Korean baseline notebook expects Korean columns
and prompts; it should be adapted before evaluating this new English pipeline.